In [1]:
from model.MF import *
from preprocess.AmazonBook import *
from evaluation.MF_evaluation import *
pd.options.display.max_rows = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
path = './dataset/amazon-book'
dataset = AmazonBook(path)

# Data(num_nodes=144242, edge_index=[2, 2380730], edge_label_index=[2, 603378])
data = dataset.get()
num_users, num_books = dataset.getNumber()
config = {
    'k': 20,
    'learning_rate': 1e-5,  # over-fitting
    'epochs': 100,
    'num_layers': 2,
    'batch_size': 8192,
    'embedding_dim': 64,
    'num_users': num_users,
    'num_books': num_books,
    'tuning_type': None,
    "weight_decay": 1e-7,
    'global_bias':(data.edge_index.size(1) + data.edge_label_index.size(1) + 2) / (num_books * num_users)
}
model = MF(
    num_users= config['num_users'],
    num_items= config['num_books'],
    mean = config['global_bias'],
    embedding_dim = config['embedding_dim']
).to(device)

# Split the dataset

In [2]:
from util.dataset_splitter import DatasetSplitter
datasplit = DatasetSplitter(dataset_name="AmazonBook")
retain_data, forget_data = datasplit.load_split()

# Retain MF model

In [3]:
retrain_model = MF(
    num_users= config['num_users'],
    num_items= config['num_books'],
    mean = config['global_bias'],
    embedding_dim = config['embedding_dim']
).to(device)
retrain_model = MF_evaluation(retrain_model, config, retain_data, device)

Epoch 1/100, Train Loss: 0.9967, HR@20: 0.0001, Recall@20: 0.0002, NDCG@20: 0.0002
Epoch 2/100, Train Loss: 0.9923, HR@20: 0.0001, Recall@20: 0.0002, NDCG@20: 0.0002
Epoch 3/100, Train Loss: 0.9880, HR@20: 0.0001, Recall@20: 0.0002, NDCG@20: 0.0002
Epoch 4/100, Train Loss: 0.9836, HR@20: 0.0001, Recall@20: 0.0002, NDCG@20: 0.0002
Epoch 5/100, Train Loss: 0.9793, HR@20: 0.0001, Recall@20: 0.0002, NDCG@20: 0.0002
Epoch 6/100, Train Loss: 0.9750, HR@20: 0.0001, Recall@20: 0.0002, NDCG@20: 0.0002
Epoch 7/100, Train Loss: 0.9707, HR@20: 0.0001, Recall@20: 0.0002, NDCG@20: 0.0002
Epoch 8/100, Train Loss: 0.9663, HR@20: 0.0001, Recall@20: 0.0002, NDCG@20: 0.0002
Epoch 9/100, Train Loss: 0.9621, HR@20: 0.0002, Recall@20: 0.0003, NDCG@20: 0.0002
Epoch 10/100, Train Loss: 0.9578, HR@20: 0.0002, Recall@20: 0.0003, NDCG@20: 0.0003
Epoch 11/100, Train Loss: 0.9535, HR@20: 0.0003, Recall@20: 0.0004, NDCG@20: 0.0004
Epoch 12/100, Train Loss: 0.9492, HR@20: 0.0004, Recall@20: 0.0006, NDCG@20: 0.0006
E

In [4]:
MF_unlearning_evaluation(retrain_model, None, forget_data, config, device)

HR@20: 0.0052, Recall@20: 0.0261, NDCG@20: 0.0154


# Prompt Unlearning
## Case 1: Without Contrastive Loss and Regularization

In [3]:
# Define the model
teacher = MF(
    num_users= config['num_users'],
    num_items= config['num_books'],
    mean = config['global_bias'],
    embedding_dim = config['embedding_dim']
).to(device)
student = MF(
    num_users= config['num_users'],
    num_items= config['num_books'],
    mean = config['global_bias'],
    embedding_dim = config['embedding_dim']
).to(device)

# Load the model
teacher.load_state_dict(torch.load(f"MF_Amazon_Book_{config['epochs']}_Epochs_Top_{config['k']}.pt"))
student.load_state_dict(torch.load(f"MF_Amazon_Book_{config['epochs']}_Epochs_Top_{config['k']}.pt"))

<All keys matched successfully>

In [4]:
# Setting the basic hyperparameters
config['alpha'] = 0.3
config['beta'] = 0.7
config['epochs'] = 20
config['gamma'] = 1e-6 # contrastive loss
config['tuning_type'] = 'simplePrompt'
config['learning_rate'] = 1e-5
config['weight_decay'] = 0
config['Contrastive_loss'] = False
student, prompt= prompt_MF_evaluation(teacher, student, config, retain_data, forget_data, device)

c:\Users\MSI\.conda\envs\master\lib\site-packages\torch_geometric\data\storage.py:450: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_label_index', 'edge_index', 'x'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  warnings.warn(


Epoch 1/20, Train Loss: 0.6686, HR@20: 0.0058, Recall@20: 0.0111, NDCG@20: 0.0098
Epoch 2/20, Train Loss: 0.6686, HR@20: 0.0058, Recall@20: 0.0111, NDCG@20: 0.0098
Epoch 3/20, Train Loss: 0.6685, HR@20: 0.0058, Recall@20: 0.0111, NDCG@20: 0.0098
Epoch 4/20, Train Loss: 0.6685, HR@20: 0.0058, Recall@20: 0.0111, NDCG@20: 0.0098
Epoch 5/20, Train Loss: 0.6684, HR@20: 0.0058, Recall@20: 0.0112, NDCG@20: 0.0098
Epoch 6/20, Train Loss: 0.6684, HR@20: 0.0058, Recall@20: 0.0112, NDCG@20: 0.0098
Epoch 7/20, Train Loss: 0.6684, HR@20: 0.0059, Recall@20: 0.0112, NDCG@20: 0.0098
Epoch 8/20, Train Loss: 0.6683, HR@20: 0.0059, Recall@20: 0.0112, NDCG@20: 0.0098
Epoch 9/20, Train Loss: 0.6683, HR@20: 0.0059, Recall@20: 0.0112, NDCG@20: 0.0098
Epoch 10/20, Train Loss: 0.6682, HR@20: 0.0059, Recall@20: 0.0112, NDCG@20: 0.0098
Epoch 11/20, Train Loss: 0.6682, HR@20: 0.0059, Recall@20: 0.0112, NDCG@20: 0.0098
Epoch 12/20, Train Loss: 0.6682, HR@20: 0.0059, Recall@20: 0.0112, NDCG@20: 0.0098
Epoch 13/20, 

In [5]:
MF_unlearning_evaluation(student, prompt, forget_data, config, device)

HR@20: 0.0038, Recall@20: 0.0176, NDCG@20: 0.0102


## Case 2: With Contrastive Loss

In [7]:
# Setting the basic hyperparameters
config['contrastive_loss'] = True
student, prompt = prompt_MF_evaluation(teacher, student, config, retain_data, forget_data, device)

KeyboardInterrupt: 

In [ ]:
MF_unlearning_evaluation(student, prompt, forget_data, config, device)

# Additional Materials(Multiple Prompts)

In [6]:
config['tuning_type'] = 'complexPrompt'
config["number_p"] = 10 # The number of prompts

student, prompt = prompt_MF_evaluation(teacher, student, config, retain_data, forget_data, device)

Epoch 1/20, Train Loss: 0.6963, HR@20: 0.0064, Recall@20: 0.0137, NDCG@20: 0.0111
Epoch 2/20, Train Loss: 0.6963, HR@20: 0.0064, Recall@20: 0.0137, NDCG@20: 0.0111
Epoch 3/20, Train Loss: 0.6963, HR@20: 0.0064, Recall@20: 0.0137, NDCG@20: 0.0111
Epoch 4/20, Train Loss: 0.6962, HR@20: 0.0064, Recall@20: 0.0137, NDCG@20: 0.0111
Epoch 5/20, Train Loss: 0.6962, HR@20: 0.0064, Recall@20: 0.0137, NDCG@20: 0.0111
Epoch 6/20, Train Loss: 0.6962, HR@20: 0.0064, Recall@20: 0.0137, NDCG@20: 0.0111
Epoch 7/20, Train Loss: 0.6962, HR@20: 0.0063, Recall@20: 0.0137, NDCG@20: 0.0110
Epoch 8/20, Train Loss: 0.6961, HR@20: 0.0063, Recall@20: 0.0137, NDCG@20: 0.0110
Epoch 9/20, Train Loss: 0.6961, HR@20: 0.0063, Recall@20: 0.0137, NDCG@20: 0.0110
Epoch 10/20, Train Loss: 0.6961, HR@20: 0.0063, Recall@20: 0.0137, NDCG@20: 0.0110
Epoch 11/20, Train Loss: 0.6960, HR@20: 0.0063, Recall@20: 0.0137, NDCG@20: 0.0110
Epoch 12/20, Train Loss: 0.6960, HR@20: 0.0063, Recall@20: 0.0137, NDCG@20: 0.0110
Epoch 13/20, 

In [7]:
MF_unlearning_evaluation(student, prompt, forget_data, config, device)

HR@20: 0.0044, Recall@20: 0.0239, NDCG@20: 0.0128
